In [5]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader
import numpy as np 
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt 

import os 
import json

from typing import List
from collections import Counter

DATA_PATH = "/Volumes/New Volume/malware-detection-dataset/opcodes/processed-data"
BATCH_SIZE = 64
MAX_LENGTH = 512
EPOCHS = 20

In [6]:
def get_data(path: os.PathLike, full_path: bool = True) -> List[str]:
    all_files = os.listdir(path)
    
    if full_path:
        return [os.path.join(path, file) for file in all_files if file.endswith('.txt')]
    else: 
        return all_files

def get_labels(filenames):
    return [1 if "VirusShare" in filename else 0 for filename in filenames]

paths = get_data(DATA_PATH)
labels = get_labels(paths)

In [7]:
# Create Vocab 
vocab = set()

loop = tqdm(paths, desc="Gathering Vocab")
for path in loop:
    with open(path, 'r') as file: 
        content = file.readlines()
        
        opcodes = [instr.rstrip() for instr in content]

        vocab.update(opcodes)
        loop.set_postfix_str(f"Size: {len(vocab)}")
vocab_size = len(vocab)

Gathering Vocab: 100%|██████████| 6940/6940 [01:17<00:00, 89.98it/s, Size: 1293] 


In [8]:
vocab_lookup = {term: idx for idx, term in enumerate(vocab)}

In [9]:
class OpcodeDataset(Dataset): 
    def __init__(self, paths, labels):
        assert len(paths) == len(labels), "Mismatch between number of files and labels"
        self.paths = paths 
        self.labels = labels

    def __len__(self):
        return len(self.paths)        


    def __getitem__(self, idx):
        assert 0 <= idx <= len(self), "Index out of range"
        label = self.labels[idx]

        with open(self.paths[idx], 'r') as file: 
            content = file.readlines() 
            
        if len(content) >= MAX_LENGTH: 
            start = torch.randint(len(content) - MAX_LENGTH, (1,))
            sequence = [vocab_lookup[instr.rstrip()] for instr in content[start:start + MAX_LENGTH]]
        else: 
            sequence = [vocab_lookup[instr.rstrip()] for instr in content]
        X = torch.full((MAX_LENGTH,), -1)
        X[:len(sequence)] = torch.tensor(sequence)
        return X, label

train_paths, test_paths, train_labels, test_labels = train_test_split(paths, labels)
train_data = OpcodeDataset(train_paths, train_labels)
test_data = OpcodeDataset(test_paths, test_labels)
train_loader = DataLoader(train_data, BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, BATCH_SIZE, shuffle=True)

In [10]:
class Classifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int, 
        num_heads: int, 
        num_layers: int, 
        hidden_dim: int, 
        max_seq_len: int, 
        dropout: float = 0.1,
    ):
        super(Classifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_encoding = nn.Parameter(torch.randn(1, max_seq_len, embedding_dim))

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embedding_dim, 
                nhead=num_heads, 
                dim_feedforward=hidden_dim, 
                dropout=dropout, 
                batch_first=True
            ), 
            num_layers
        )

        self.fc = nn.Linear(embedding_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        x = self.embedding(x) + self.position_encoding[:, :x.shape[1], :]
        x = self.encoder(x) 
        x = x.mean(dim=1)
        x = self.dropout(x)
        return self.fc(x)

In [11]:

device = torch.device('mps')

model = Classifier(
    vocab_size=vocab_size, embedding_dim=128, num_heads=4, num_layers=2, 
    hidden_dim=256, max_seq_len=MAX_LENGTH,
).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(model.parameters(), lr=2e-4)

train_loss = []
test_loss = []
test_acc = []

def train(model, data_loader, loss_fn, optim):
    train_loop = tqdm(data_loader, desc=f"Train {epoch + 1}/{EPOCHS}", unit="batch", leave=False)

    for sequence, label in train_loop: 
        sequence = sequence.to(device)
        label = label.float().to(device)

        optim.zero_grad() 
        output = model(sequence).squeeze(1)
        
        loss = loss_fn(output, label)
        
        loss.backward() 
        optim.step() 

        train_loop.set_postfix_str(f"Loss: {loss.item():.2f}")
        train_loss.append(loss.item())

def test(model, data_loader, loss_fn):
    total_loss = 0.0
    total_correct = 0 
    total = 0 
    model.eval() 
    
    for sequence, label in tqdm(data_loader, desc=f"Test {epoch + 1}/{EPOCHS}", leave=False, unit="batch"):
        sequence = sequence.to(device)
        label = label.float().to(device)

        with torch.no_grad():
            output = model(sequence).squeeze(1)

        loss = loss_fn(output, label)

        total_loss += loss.item() 
        predictions = (torch.sigmoid(output) > 0.5).long() 
        total_correct += (predictions == label.long()).sum().item() 
        total += label.size(0)
        
    acc = total_correct / total 
    test_loss.append(total_loss / len(data_loader))
    test_acc.append(acc)
    print(f"Epoch {epoch + 1}/{EPOCHS}\nLoss: {total_loss / len(data_loader)}, Acc: {acc:.2f}")
    

for epoch in range(EPOCHS):
    train(model, train_loader, loss_fn, optim)
    test(model, test_loader, loss_fn)

Epoch 1/20
Loss: 0.47730787311281475, Acc: 0.77

Epoch 2/20
Loss: 0.4530185940010207, Acc: 0.77

Epoch 3/20
Loss: 0.44470525107213427, Acc: 0.78

Epoch 4/20
Loss: 0.3941628805228642, Acc: 0.80

Epoch 5/20
Loss: 0.39304635354450773, Acc: 0.82

...

Epoch 16/20
Loss: 0.3513257641877447, Acc: 0.84

Epoch 17/20
Loss: 0.33291883899697233, Acc: 0.84

Epoch 18/20
Loss: 0.35033791459032465, Acc: 0.83

Epoch 19/20
Loss: 0.336183919438294, Acc: 0.84

Epoch 20/20
Loss: 0.3425631123994078, Acc: 0.84

In [16]:
vocab_size

1293